## Imports & setting up:

In [ ]:
import os

os.environ["PYKEOPS_VERBOSE"] = "0"
os.environ["KEOPS_VERBOSE"] = "0"

In [ ]:
import sys
sys.path.append(os.path.join(os.getcwd(), '../../')) # Add root of repo to import MBM

import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from cmcrameri import cm
import massbalancemachine as mbm
import logging
import torch 
from matplotlib.colors import LinearSegmentedColormap
import matplotlib as mpl
from matplotlib.ticker import FuncFormatter
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colorbar as mcolorbar

from matplotlib.lines import Line2D
import scipy.stats as scipy_stats
from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.dataset import *
from regions.TF_Europe.scripts.plotting import *
from regions.TF_Europe.scripts.models import *
from regions.TF_Europe.scripts.models import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

# Initialize logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
BASE_DIR = Path(cfg.dataPath) / path_cache / f"CROSS_REGION"
print(f"Base directory for data: {BASE_DIR}")

MONTHLY_COLS = [
    't2m',
    'tp',
    'slhf',
    'sshf',
    'ssrd',
    'fal',
    'str',
    'ELEVATION_DIFFERENCE',
]
STATIC_COLS = ['aspect', 'slope', 'svf']

feature_columns = MONTHLY_COLS + STATIC_COLS

# --- plot KDEs ---
COL_LABELS = {
    "t2m": "2m temperature",
    "tp": "Precipitation",
    "slhf": "Latent heat flux",
    "sshf": "Sensible heat flux",
    "ssrd": "Solar radiation",
    "fal": "Albedo",
    "str": "Thermal radiation",
    "ELEVATION_DIFFERENCE": "Elevation diff. (m)",
    "aspect": "Aspect (°)",
    "slope": "Slope (°)",
    "svf": "Sky-view factor",
}

# Transform data to monthly format (run or load data):
paths = {
    'era5_climate_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_monthly_averaged_data_EU_US_CANADA.nc"),
    'geopotential_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_geopotential_pressure_EU_US_CANADA.nc")
}

# Check that all these files exists
for key, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file for {key} not found at {path}")

nature_seq = LinearSegmentedColormap.from_list(
    "nature_seq",
    [
        NATURE_PALETTE["sky_blue"], NATURE_PALETTE["blue"],
        NATURE_PALETTE["black"]
    ],
)


## Load glacier grids:

In [ ]:
"""
Examples of loading data:
# Load Switzerland only
df = load_stakes(cfg, "CH")

# Load all Central Europe (FR+CH+IT+AT when you add them)
df_ceu = load_stakes_for_rgi_region(cfg, "11")

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}"""

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}
dfs["11"].head(2)

### Add subregions:

In [ ]:
# Central Asia
dfs, glacier_dict_ca = add_o2region_to_dfs(dfs, '13', cfg)

# Alaska
dfs, glacier_dict_alaska = add_o2region_to_dfs(dfs,
                                               '01',
                                               cfg,
                                               deduplicate_glaciers=True)

# Italy:
df_ceu = dfs['11']
df_ceu.loc[df_ceu["GLACIER"].isin(IT_GLACIERS), "SOURCE_CODE"] = "IT"
df_ceu.loc[df_ceu["GLACIER"].isin(AT_GLACIERS), "SOURCE_CODE"] = "AT"
dfs['11'] = df_ceu

## Maps

In [ ]:
def build_region_glacier_info(
    cfg,
    *,
    rgi_region_id: str,
    outline_shp_path: str,
    glacier_col="GLACIER",
    lat_col="POINT_LAT",
    lon_col="POINT_LON",
    period_col="PERIOD",
    nmeas_col="Nb. measurements",
    source_col="SOURCE_CODE",
    source_resolution="mode",
    o2region_col="O2Region",
    load_stakes_fn=None,
    verbose=True,
):
    """
    Generic builder for per-glacier info tables (for maps / summaries), for any region.
    No FT/holdout/train-test labeling — just location, measurement counts, source,
    and (if present) O2 sub-region (Central Asia / Alaska) or country override
    (Central Europe: IT/AT split via IT_GLACIERS/AT_GLACIERS).
    """

    if load_stakes_fn is None:
        load_stakes_fn = load_stakes_for_rgi_region  # noqa: F821

    data_region = load_stakes_fn(cfg, rgi_region_id)

    # --- O2Region split (Central Asia / Alaska only) ---
    if rgi_region_id == "13":
        dfs, glacier_dict = add_o2region_to_dfs({rgi_region_id: data_region},
                                                "13", cfg)
        data_region = dfs[rgi_region_id]
    elif rgi_region_id == "01":
        dfs, glacier_dict = add_o2region_to_dfs({rgi_region_id: data_region},
                                                "01",
                                                cfg,
                                                deduplicate_glaciers=True)
        data_region = dfs[rgi_region_id]

    # --- Country override (Central Europe only: IT / AT) ---
    elif rgi_region_id == "11":
        data_region.loc[data_region[glacier_col].isin(IT_GLACIERS),
                        source_col] = "IT"  # noqa: F821
        data_region.loc[data_region[glacier_col].isin(AT_GLACIERS),
                        source_col] = "AT"  # noqa: F821
        if verbose:
            n_it = data_region[source_col].eq("IT").sum()
            n_at = data_region[source_col].eq("AT").sum()
            print(
                f"[{rgi_region_id}] Overrode {source_col}: IT -> {n_it} stake rows, "
                f"AT -> {n_at} stake rows.")

    glacier_outline_rgi = gpd.read_file(outline_shp_path)

    if verbose:
        print(f"[{rgi_region_id}] stake rows: {len(data_region)} | "
              f"glaciers: {data_region[glacier_col].nunique()}")

    # --- measurement counts ---
    meas_counts = (data_region.groupby(glacier_col).size().sort_values(
        ascending=False).rename(nmeas_col).to_frame())

    # --- mean location ---
    glacier_loc = data_region.groupby(glacier_col)[[lat_col, lon_col]].mean()

    # --- counts per period (winter/annual) ---
    glacier_period = (data_region.groupby(
        [glacier_col, period_col]).size().unstack().fillna(0).astype(int))

    # --- SOURCE_CODE per glacier ---
    if source_col in data_region.columns:
        gsrc = data_region.groupby(glacier_col)[source_col].apply(
            lambda s: s.dropna().astype(str).unique())

        mixed = gsrc[gsrc.apply(len) > 1]
        if len(mixed) > 0 and verbose:
            print(
                f"Warning: {len(mixed)} glaciers have multiple {source_col} values "
                f"(showing up to 5): {mixed.head(5).to_dict()}")

        if len(mixed) > 0 and source_resolution == "error":
            raise ValueError(
                f"Found glaciers with multiple {source_col} values. "
                f"Set source_resolution to 'first', 'mode', or 'list' to resolve."
            )

        if source_resolution == "list":
            glacier_source = (gsrc.apply(lambda arr: list(arr)).rename(
                source_col).to_frame())
        elif source_resolution == "first":
            glacier_source = (gsrc.apply(lambda arr: arr[0] if len(arr) else
                                         None).rename(source_col).to_frame())
        elif source_resolution == "mode":

            def _mode(series):
                s = series.dropna().astype(str)
                if len(s) == 0:
                    return None
                return s.value_counts().index[0]

            glacier_source = (data_region.groupby(glacier_col)[source_col].
                              apply(_mode).rename(source_col).to_frame())
        else:
            raise ValueError(
                "source_resolution must be one of: 'error','first','mode','list'"
            )
    else:
        glacier_source = None
        if verbose:
            print(
                f"Note: '{source_col}' not found in data_region; skipping SOURCE_CODE merge."
            )

    # --- O2Region per glacier (mode, in case of any inconsistency) ---
    if o2region_col in data_region.columns:

        def _mode(series):
            s = series.dropna().astype(str)
            if len(s) == 0:
                return None
            return s.value_counts().index[0]

        gregion = data_region.groupby(glacier_col)[o2region_col].apply(_mode)

        n_unique_per_glacier = data_region.groupby(
            glacier_col)[o2region_col].nunique()
        inconsistent = n_unique_per_glacier[n_unique_per_glacier > 1]
        if len(inconsistent) > 0 and verbose:
            print(
                f"Warning: {len(inconsistent)} glaciers have multiple {o2region_col} "
                f"values (using mode): {list(inconsistent.index[:5])}")

        glacier_o2region = gregion.rename(o2region_col).to_frame()
    else:
        glacier_o2region = None
        if verbose:
            print(
                f"Note: '{o2region_col}' not found in data_region; skipping O2Region merge."
            )

    # --- merge base ---
    glacier_info = glacier_loc.join(meas_counts,
                                    how="inner").join(glacier_period,
                                                      how="left")
    if glacier_source is not None:
        glacier_info = glacier_info.join(glacier_source, how="left")
    if glacier_o2region is not None:
        glacier_info = glacier_info.join(glacier_o2region, how="left")

    return data_region, glacier_outline_rgi, glacier_info

In [ ]:
from regions.TF_Europe.scripts.plotting.style import NATURE_COLORS, nature_figsize, NATURE_SPECS


def plot_glacier_measurements_map(
        glacier_info,
        glacier_outline_rgi,
        *,
        lon_col="POINT_LON",
        lat_col="POINT_LAT",
        nmeas_col="Nb. measurements",
        hue_col=None,
        palette=None,
        cmap_for_hue=None,
        title="Glacier measurement locations",
        figsize=None,
        nature_cols=1,
        nature_height_mm=80,
        extent=(5.8, 15, 44, 48),
        use_size=True,  # : NEW — if False, plot fixed-size points (location only), skip size legend
        fixed_size=80,  # : NEW — marker size (points^2) used when use_size=False
        sizes=(20, 300),
        size_legend_values=(30, 100, 1000, 6000),
        alpha=0.6,
        point_color=NATURE_COLORS[5],
        land_facecolor="lightgray",
        land_alpha=0.5,
        add_features=True,
        add_gridlines=True,
        legend_loc="lower right",
        legend_ncol=2,
        legend_fontsize=NATURE_SPECS["font_max_pt"],
        legend_title_fontsize=NATURE_SPECS["font_max_pt"],
        hue_legend_title=None,
        gridline_label_fontsize=NATURE_SPECS["font_max_pt"] - 1,
        zorder_scatter=10,
        data_region=None,
        year_col="YEAR",
        summary_loc=(0.02, 0.02),
        summary_fontsize=NATURE_SPECS["font_max_pt"],
        legend_separate=False,
        legend_figsize=(3, 2),
        show=True,
):
    """
    Plot glacier point locations on a Cartopy map, styled to Nature figure specs
    (font sizes, dpi, colors via NATURE_COLORS).

    If use_size=True (default), marker size ~ sqrt(# measurements), with a size legend.
    If use_size=False, all points are drawn at a fixed size (location only) — no
    size legend is built, useful for a simpler "where are the glaciers" map.

    If hue_col is given, points are colored by that categorical column (e.g. O2Region)
    using NATURE_COLORS by default (or a custom palette / cmap_for_hue), and a combined
    hue(+size, if use_size) legend is built. Otherwise, single color for all points
    (legend then only shows hue, or nothing if hue_col is also None and use_size=False).

    If legend_separate=True, the legend is NOT drawn on the map axes; instead a second,
    standalone Figure containing only the legend is created and returned, so it can be
    saved independently (e.g. for separate placement in Illustrator).

    Requires (in your environment):
      - numpy as np
      - matplotlib.pyplot as plt
      - seaborn as sns
      - cartopy.crs as ccrs
      - cartopy.feature as cfeature
      - from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
      - matplotlib.lines import Line2D
      - utils.plotting.style: NATURE_COLORS, nature_figsize, NATURE_SPECS

    Returns
    -------
    If legend_separate=False (default):
        fig, ax, glacier_info_out, scaled_size_fn
    If legend_separate=True:
        fig, ax, glacier_info_out, scaled_size_fn, legend_fig

    Note: scaled_size_fn is a no-op-ish passthrough wrapper when use_size=False
    (returns fixed_size regardless of input), kept for return-signature consistency.
    """

    if figsize is None:
        figsize = nature_figsize(cols=nature_cols, height_mm=nature_height_mm)

    # --- 0) Copy input ---
    df = glacier_info.copy()

    # --- 1) Preprocessing: sqrt scaling (only if use_size) ---
    if use_size:
        if nmeas_col not in df.columns:
            raise KeyError(f"Missing '{nmeas_col}' in glacier_info.")
        df["sqrt_size"] = np.sqrt(df[nmeas_col].astype(float))

        sqrt_min = float(df["sqrt_size"].min())
        sqrt_max = float(df["sqrt_size"].max())

        def scaled_size(val, min_out=sizes[0], max_out=sizes[1]):
            sqrt_val = np.sqrt(float(val))
            if sqrt_max == sqrt_min:
                return (min_out + max_out) / 2
            return min_out + (max_out - min_out) * ((sqrt_val - sqrt_min) /
                                                    (sqrt_max - sqrt_min))

        df["scaled_size"] = df[nmeas_col].apply(scaled_size)
        size_arg = "scaled_size"
        size_kwarg = dict(size="scaled_size", sizes=sizes)
    else:

        def scaled_size(val, min_out=fixed_size, max_out=fixed_size):
            return fixed_size

        df["scaled_size"] = fixed_size
        size_kwarg = dict(s=fixed_size)  # fixed marker size, no size mapping

    # --- 1b) Hue setup ---
    use_hue = hue_col is not None
    if use_hue:
        if hue_col not in df.columns:
            raise KeyError(f"Missing '{hue_col}' in glacier_info.")
        df[hue_col] = df[hue_col].astype(str)
        categories = sorted(df[hue_col].dropna().unique())

        if palette is None:
            if cmap_for_hue is not None:
                try:
                    colors = get_cmap_hex(cmap_for_hue,
                                          len(categories))  # noqa: F821
                except Exception:
                    colors = NATURE_COLORS[:len(categories)]
            else:
                if len(categories) > len(NATURE_COLORS):
                    print(
                        f"Warning: {len(categories)} categories but only "
                        f"{len(NATURE_COLORS)} NATURE_COLORS available; colors will repeat."
                    )
                colors = [
                    NATURE_COLORS[i % len(NATURE_COLORS)]
                    for i in range(len(categories))
                ]
            palette = dict(zip(categories, colors))
        if hue_legend_title is None:
            hue_legend_title = hue_col

    # --- 2) Figure + map ---
    fig = plt.figure(figsize=figsize)

    lonW, lonE, latS, latN = extent

    max_lat, min_lat = float(df[lat_col].max()), float(df[lat_col].min())
    max_lon, min_lon = float(df[lon_col].max()), float(df[lon_col].min())

    if not (latS <= min_lat <= max_lat <= latN):
        print(
            f"Warning: latitude bounds may not fully contain glacier points: "
            f"glacier lat range [{min_lat:.2f}, {max_lat:.2f}] vs map bounds [{latS}, {latN}]"
        )
    if not (lonW <= min_lon <= max_lon <= lonE):
        print(
            f"Warning: longitude bounds may not fully contain glacier points: "
            f"glacier lon range [{min_lon:.2f}, {max_lon:.2f}] vs map bounds [{lonW}, {lonE}]"
        )

    projPC = ccrs.PlateCarree()
    ax = plt.axes(projection=projPC)
    ax.set_extent([lonW, lonE, latS, latN], crs=ccrs.Geodetic())

    if add_features:
        ax.add_feature(cfeature.COASTLINE, linewidth=0.4)
        ax.add_feature(cfeature.LAKES, linewidth=0.3)
        ax.add_feature(cfeature.RIVERS, linewidth=0.3)
        ax.add_feature(cfeature.BORDERS, linestyle="-", linewidth=0.4)
        ax.add_feature(cfeature.LAND,
                       facecolor=land_facecolor,
                       alpha=land_alpha)

    glacier_outline_rgi.plot(ax=ax,
                             transform=projPC,
                             color="black",
                             alpha=0.7,
                             linewidth=0.4)

    # --- 3) Scatter ---
    if use_hue:
        g = sns.scatterplot(
            data=df,
            x=lon_col,
            y=lat_col,
            hue=hue_col,
            alpha=alpha,
            palette=palette,
            transform=projPC,
            ax=ax,
            zorder=zorder_scatter,
            legend=True,
            **size_kwarg,
        )
    else:
        g = sns.scatterplot(
            data=df,
            x=lon_col,
            y=lat_col,
            alpha=alpha,
            color=point_color,
            transform=projPC,
            ax=ax,
            zorder=zorder_scatter,
            legend=False,
            **size_kwarg,
        )

    # --- 4) Gridlines ---
    if add_gridlines:
        gl = ax.gridlines(
            draw_labels=True,
            linewidth=0.3,
            color="gray",
            alpha=0.5,
            linestyle="--",
        )
        gl.xformatter = LONGITUDE_FORMATTER
        gl.yformatter = LATITUDE_FORMATTER
        gl.xlabel_style = {"size": gridline_label_fontsize, "color": "black"}
        gl.ylabel_style = {"size": gridline_label_fontsize, "color": "black"}
        gl.top_labels = gl.right_labels = False

    # --- 5) Legend handles ---
    if use_size:
        size_handles = [
            Line2D(
                [],
                [],
                marker="o",
                linestyle="None",
                markersize=np.sqrt(scaled_size(val)),
                markerfacecolor=(point_color if not use_hue else "gray"),
                alpha=alpha,
                label=f"{val}",
            ) for val in size_legend_values
        ]
    else:
        size_handles = []

    if use_hue:
        handles, labels = g.get_legend_handles_labels()
        expected_labels = list(palette.keys())
        label_to_handle = dict(zip(labels, handles))
        hue_entries = [(label_to_handle[l], l) for l in expected_labels
                       if l in label_to_handle]

        combined_handles = [h for h, _ in hue_entries] + size_handles
        combined_labels = [l for _, l in hue_entries
                           ] + [str(v) for v in size_legend_values]
        legend_title = f"{hue_legend_title} / Number of measurements" if use_size else hue_legend_title
    else:
        combined_handles = size_handles
        combined_labels = [str(v) for v in size_legend_values]
        legend_title = "Number of measurements"

    # Remove seaborn's auto-attached hue legend (we build our own combined one, if any)
    if use_hue and g.legend_ is not None:
        g.legend_.remove()

    legend_fig = None
    has_legend_content = len(combined_handles) > 0

    if legend_separate:
        if has_legend_content:
            legend_fig = plt.figure(figsize=legend_figsize)
            legend_fig.legend(
                combined_handles,
                combined_labels,
                title=legend_title,
                loc="center",
                frameon=True,
                fontsize=legend_fontsize,
                title_fontsize=legend_title_fontsize,
                borderpad=0.8,
                labelspacing=0.8,
                ncol=legend_ncol,
            )
            legend_fig.gca().set_axis_off() if legend_fig.axes else None
    elif has_legend_content:
        leg = ax.legend(
            combined_handles,
            combined_labels,
            title=legend_title,
            loc=legend_loc,
            frameon=True,
            fontsize=legend_fontsize,
            title_fontsize=legend_title_fontsize,
            borderpad=0.8,
            labelspacing=0.8,
            ncol=legend_ncol,
        )
        leg.get_frame().set_linewidth(0.4)

    # --- 6) Summary text box ---
    if data_region is not None:
        n_glaciers = len(glacier_info.index.unique())

        if year_col in data_region.columns:
            years = data_region[year_col].dropna()
            if len(years) > 0:
                y_min, y_max = int(years.min()), int(years.max())
                year_str = f"{y_min}\u2013{y_max}" if y_min != y_max else f"{y_min}"
            else:
                year_str = "N/A"
        else:
            print(
                f"Note: '{year_col}' not found in data_region; skipping years in summary."
            )
            year_str = "N/A"

        summary_text = f"Glaciers: {n_glaciers}\nYears: {year_str}"
        print(summary_text)

    if title:
        ax.set_title(title, fontsize=NATURE_SPECS["font_max_pt"])

    plt.tight_layout()

    if show:
        plt.show()

    if legend_separate:
        return fig, ax, df, scaled_size, legend_fig
    return fig, ax, df, scaled_size

### CEU:

In [ ]:
data_CEU, glacier_outline_CEU, glacier_df_CEU = build_region_glacier_info(
    cfg,
    rgi_region_id="11",
    outline_shp_path=cfg.dataPath +
    "RGI_v6/RGI_11_CentralEurope/11_rgi60_CentralEurope.shp",
)
# IT/AT override now happens automatically inside the function

palette_ceu = {
    "CH": NATURE_PALETTE["blue"],
    "FR": NATURE_PALETTE["orange"],
    "IT": NATURE_PALETTE["bluish_green"],
    "AT": NATURE_PALETTE["reddish_purple"],
}

fig, ax, glacier_info_plot, scaled_size_fng, leg = plot_glacier_measurements_map(
    glacier_info=glacier_df_CEU,
    data_region=data_CEU,
    glacier_outline_rgi=glacier_outline_CEU,
    title="Glacier measurement locations Central European Alps",
    extent=(5.8, 15, 44, 48),
    # sizes=(20, 800),
    # size_legend_values=(30, 100, 1000),
    use_size=False,
    fixed_size=300,
    figsize=(10, 8),
    hue_col="SOURCE_CODE",
    hue_legend_title="Country",
    palette=palette_ceu,
    legend_separate=True)
# save figure
fig.savefig('figures/paperTF/fig_maps_CEU_gl.pdf')
leg.savefig("figures/paperTF/fig_maps_CEU_gl_legend.pdf",
            dpi=300,
            bbox_inches="tight")

### NOR:

In [ ]:
data_NOR, glacier_outline_rgi, glacier_df_NOR = build_region_glacier_info(
    cfg,
    rgi_region_id="08",
    outline_shp_path=cfg.dataPath +
    "RGI_v6/RGI_08_Scandinavia/08_rgi60_Scandinavia.shp",
)

fig, ax, glacier_info_plot, scaled_size_fn = plot_glacier_measurements_map(
    glacier_info=glacier_df_NOR,
    data_region=data_NOR,
    glacier_outline_rgi=glacier_outline_rgi,
    title="Glacier measurement locations Norway",
    extent=(4, 24, 57, 71),
    # sizes=(20, 800),
    # size_legend_values=(30, 100, 1000),
    # legend_separate=True,
    use_size=False,
    fixed_size=200,
    figsize=(10, 8),
    legend_separate=False,
    gridline_label_fontsize=10)

# save figure
fig.savefig('figures/paperTF/fig_maps_NOR_gl.pdf')

### Iceland:

In [ ]:
data_ISL, glacier_outline_rgi, glacier_df_ISL = build_region_glacier_info(
    cfg,
    rgi_region_id="06",
    outline_shp_path=cfg.dataPath +
    "RGI_v6/RGI_06_Iceland/06_rgi60_Iceland.shp",
)

fig, ax, glacier_info_plot, scaled_size_fn = plot_glacier_measurements_map(
    glacier_info=glacier_df_ISL,
    data_region=data_ISL,
    glacier_outline_rgi=glacier_outline_rgi,
    title="Glacier measurement locations Iceland",
    extent=(-25, -12, 63, 67),
    figsize=(10, 8),
    # sizes=(300, 800),
    # size_legend_values=(30, 100, 1000),
    # legend_separate=True
    use_size=False,
    fixed_size=300,
    legend_separate=False)

# save figure
fig.savefig('figures/paperTF/fig_maps_ISL_gl.pdf')

### Svalbard:

In [ ]:
data_SJM, glacier_outline_rgi, glacier_df_SJM = build_region_glacier_info(
    cfg,
    rgi_region_id="07",
    outline_shp_path=cfg.dataPath +
    "RGI_v6/RGI_07_Svalbard/07_rgi60_Svalbard.shp",
)

fig, ax, glacier_info_plot, scaled_size_fn = plot_glacier_measurements_map(
    glacier_info=glacier_df_SJM,
    data_region=data_SJM,
    glacier_outline_rgi=glacier_outline_rgi,
    title="Glacier measurement locations Svalbard",
    extent=(10, 21, 76.5, 80),
    # sizes=(300, 800),
    # size_legend_values=(30, 100, 1000),
    figsize=(18, 10),
    use_size=False,
    fixed_size=500,
    legend_separate=False)

# save figure
fig.savefig('figures/paperTF/fig_maps_SJM_gl.pdf')

### Alaska:

In [ ]:
NATURE_PALETTE

In [ ]:
data_ALA, glacier_outline_rgi, glacier_df_ALA = build_region_glacier_info(
    cfg,
    rgi_region_id="01",
    outline_shp_path=cfg.dataPath + "RGI_v6/RGI_01_Alaska/01_rgi60_Alaska.shp",
)

palette_ala = {
    "2": NATURE_PALETTE["orange"],
    "4": NATURE_PALETTE["reddish_purple"],
    "6": NATURE_PALETTE["bluish_green"],
}

fig, ax, glacier_info_plot, scaled_size_fn = plot_glacier_measurements_map(
    glacier_info=glacier_df_ALA,
    data_region=data_ALA,
    glacier_outline_rgi=glacier_outline_rgi,
    title="Glacier measurement locations Alaska",
    extent=(-170, -130, 55, 67),
    # sizes=(50, 800),
    # size_legend_values=(5, 50, 500),
    figsize=(10, 8),
    #hue_col='O2Region',
    legend_loc='upper right',
    palette=palette_ala,
    use_size=False,
    fixed_size=350,
    legend_separate=False)

# save figure
fig.savefig('figures/paperTF/fig_maps_ALA_gl.pdf')

### Central Asia:

In [ ]:
data_CA, glacier_outline_rgi, glacier_df_CA = build_region_glacier_info(
    cfg,
    rgi_region_id="13",
    outline_shp_path=cfg.dataPath +
    "RGI_v6/RGI_13_CentralAsia/13_rgi60_CentralAsia.shp",
)

palette_ca = {
    "1": NATURE_PALETTE["orange"],
    "2": NATURE_PALETTE["vermillion"],
    "3": NATURE_PALETTE["bluish_green"],
    "4": NATURE_PALETTE["reddish_purple"],
}

fig, ax, glacier_info_plot, scaled_size_fn, legend_fig = plot_glacier_measurements_map(
    glacier_info=glacier_df_CA,
    data_region=data_CA,
    glacier_outline_rgi=glacier_outline_rgi,
    title="Glacier measurement locations Central Asia",
    extent=(67, 90, 35, 46),
    # sizes=(100, 1000),
    # size_legend_values=(30, 500, 1000),
    figsize=(10, 8),
    hue_col='O2Region',
    hue_legend_title='RGI sub-region',
    legend_loc='lower right',
    palette=palette_ca,
    use_size=False,
    fixed_size=300,
    gridline_label_fontsize=8,
    legend_separate=True)

# save figure
fig.savefig('figures/paperTF/fig_maps_CA_gl.pdf')
legend_fig.savefig("figures/paperTF/fig_maps_CA_legend_color.pdf",
                   dpi=300,
                   bbox_inches="tight")

In [ ]:
test_df_preds_ca3 = pd.read_csv('test_df_preds_ca3.csv')
CA3_KEY = "CA_3_GL"  # or "CA_3_ID" — whichever exists in models_tf

color_by_glacier_ca3 = {
    'BARKRAK SREDNIY': '#000000',
    'BARKRAK SREDNIY EAST BRANCH': '#e69f00',
    'GLACIER NO. 354 (AKSHIYRAK)': '#56b4e9',
    'GLACIER NO. 599 (KJUNGEI ALA-TOO)': '#009e73',
    'TURGEN-AKSUU': '#f0e442'
}

# ── Map: same glaciers, same colors ─────────────────────────────────────────
# Match test_df_preds_ca3's GLACIER naming to glacier_df_CA's index/column.
# Adjust the column name below if glacier_df_CA doesn't use "GLACIER".
# GLACIER_COL_MAP = "GLACIER" if "GLACIER" in glacier_df_CA.columns else glacier_df_CA.index.name

ca3_glacier_names = set(test_df_preds_ca3["GLACIER"].unique())
GLACIER_COL_MAP = 'GLACIER'
if GLACIER_COL_MAP == glacier_df_CA.index.name:
    ca3_glaciers = glacier_df_CA.loc[glacier_df_CA.index.isin(
        ca3_glacier_names)].copy()
    ca3_glaciers["GLACIER"] = ca3_glaciers.index
else:
    ca3_glaciers = glacier_df_CA.loc[glacier_df_CA[GLACIER_COL_MAP].isin(
        ca3_glacier_names)].copy()
    ca3_glaciers = ca3_glaciers.rename(columns={GLACIER_COL_MAP: "GLACIER"})

missing_in_map = ca3_glacier_names - set(ca3_glaciers["GLACIER"].unique())
if missing_in_map:
    print(
        f"  ⚠ {len(missing_in_map)} glacier(s) from the pred-vs-truth plot not "
        f"found in glacier_df_CA (naming mismatch?): {missing_in_map}")

if "GLACIER" in data_CA.columns:
    data_ca3 = data_CA[data_CA["GLACIER"].isin(ca3_glacier_names)].copy()
else:
    data_ca3 = data_CA

fig_map, ax_map, glacier_info_plot, scaled_size_fn, legend_fig = plot_glacier_measurements_map(
    glacier_info=ca3_glaciers,
    data_region=data_ca3,
    glacier_outline_rgi=glacier_outline_rgi,
    title="Test glacier measurement locations — Central Asia 3",
    extent=(70, 80, 39,
            44),  # placeholder — narrow based on any warning it prints
    figsize=(8, 8),
    hue_col="GLACIER",
    hue_legend_title="Glacier",
    palette=color_by_glacier_ca3,
    use_size=False,
    fixed_size=300,
    gridline_label_fontsize=NATURE_SPECS["font_min_pt"],
    legend_separate=True,
)

fig_map.savefig('figures/paperTF/appendix/app_maps_CA3_gl_by_glacier.png')

### World plot:

In [ ]:
def plot_wgms_world_map(
        glacier_info,
        *,
        nmeas_col="Nb. measurements",
        lon_col="POINT_LON",
        lat_col="POINT_LAT",
        figsize=(16, 8),
        sizes=(100, 800),
        size_legend_values=(10, 100, 500),
        point_color="#b2182b",
        alpha=0.9,
        ocean_color="#cce5f0",
        land_color="#f0f0f0",
        legend_fontsize=NATURE_SPECS["font_max_pt"],
        legend_title_fontsize=NATURE_SPECS["font_max_pt"],
        legend_loc="lower center",
        legend_ncol=3,
        legend_separate=False,
        legend_figsize=(4, 1.5),
        show=True,
        title='Number of point mass balance observations in the WGMS database'
):
    """
    Plot a global Robinson-projection map of glacier measurement locations,
    with bubble size proportional to sqrt(# measurements). No glacier outlines.
    """
    df = glacier_info.copy()

    if nmeas_col not in df.columns:
        raise KeyError(f"Missing '{nmeas_col}' in glacier_info.")

    # --- Size scaling ---
    df["sqrt_size"] = np.sqrt(df[nmeas_col].astype(float))
    sqrt_min, sqrt_max = df["sqrt_size"].min(), df["sqrt_size"].max()

    def scaled_size(val):
        sqrt_val = np.sqrt(float(val))
        if sqrt_max == sqrt_min:
            return (sizes[0] + sizes[1]) / 2
        return sizes[0] + (sizes[1] - sizes[0]) * ((sqrt_val - sqrt_min) /
                                                   (sqrt_max - sqrt_min))

    df["scaled_size"] = df[nmeas_col].apply(scaled_size)

    # --- Figure ---
    projRob = ccrs.Robinson()
    projPC = ccrs.PlateCarree()

    fig = plt.figure(figsize=figsize)
    ax = plt.axes(projection=projRob)
    ax.set_global()

    ax.add_feature(cfeature.OCEAN, facecolor=ocean_color, zorder=0)
    ax.add_feature(cfeature.LAND, facecolor=land_color, zorder=1)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.3, zorder=2)
    ax.add_feature(cfeature.BORDERS, linewidth=0.2, zorder=2)
    ax.spines["geo"].set_linewidth(0.8)

    # --- Scatter ---
    sns.scatterplot(
        data=df,
        x=lon_col,
        y=lat_col,
        size="scaled_size",
        sizes=sizes,
        color=point_color,
        alpha=alpha,
        transform=projPC,
        ax=ax,
        zorder=3,
        legend=False,
    )

    # --- Legend ---
    size_handles = [
        Line2D(
            [],
            [],
            marker="o",
            linestyle="None",
            markersize=np.sqrt(scaled_size(v)),
            markerfacecolor=point_color,
            markeredgewidth=0,
            alpha=alpha,
            label=str(v),
        ) for v in size_legend_values
    ]

    legend_fig = None
    if legend_separate:
        legend_fig = plt.figure(figsize=legend_figsize)
        legend_fig.legend(
            size_handles,
            [str(v) for v in size_legend_values],
            title="Measurements",
            loc="center",
            frameon=True,
            fontsize=legend_fontsize,
            title_fontsize=legend_title_fontsize,
            borderpad=0.8,
            labelspacing=0.8,
            ncol=legend_ncol,
        )
        if legend_fig.axes:
            legend_fig.gca().set_axis_off()
    else:
        leg = ax.legend(
            size_handles,
            [str(v) for v in size_legend_values],
            title="Number of observations per glacier",
            loc=legend_loc,
            frameon=True,
            fontsize=legend_fontsize,
            title_fontsize=legend_title_fontsize,
            borderpad=0.8,
            labelspacing=0.8,
            ncol=legend_ncol,
        )
        leg.get_frame().set_linewidth(0.4)

    ax.set_title(title, fontsize=NATURE_SPECS["font_max_pt"] + 2)

    plt.tight_layout()
    if show:
        plt.show()

    if legend_separate:
        return fig, ax, df, legend_fig

    return fig, ax, df

In [ ]:
def build_wgms_global_glacier_info(cfg, nmeas_col="Nb. measurements"):
    """
    Load full WGMS mass_balance_point.csv and build a per-glacier info table
    for the world map. Lat/lon come directly from stake rows (already in df_stakes).
    """
    df_stakes = pd.read_csv(
        os.path.join(cfg.dataPath, "WGMS/raw/mass_balance_point.csv"))
    df_rgi = pd.read_csv(os.path.join(cfg.dataPath, "WGMS/raw/glacier.csv"))

    # --- mappings from glacier.csv ---
    id_to_rgi = dict(zip(df_rgi["id"], df_rgi["rgi60_ids"]))
    id_to_gtng = dict(zip(df_rgi["id"], df_rgi["gtng_region"]))
    df_stakes["RGIId"] = df_stakes["glacier_id"].map(id_to_rgi)
    df_stakes["gtng_region"] = df_stakes["glacier_id"].map(id_to_gtng)

    # --- rename to MBM convention ---
    df_stakes = df_stakes.rename(
        columns={
            "latitude": "POINT_LAT",
            "longitude": "POINT_LON",
            "elevation": "POINT_ELEVATION",
            "begin_date": "FROM_DATE",
            "end_date": "TO_DATE",
            "balance": "POINT_BALANCE",
            "glacier_name": "GLACIER",
            "year": "YEAR",
            "balance_code": "PERIOD",
            "gtng_region": "RGI_REGION",
        })

    # --- drop rows missing coordinates ---
    df_stakes = df_stakes.dropna(subset=["POINT_LAT", "POINT_LON"])

    # --- per-glacier aggregation ---
    glacier_info = (df_stakes.groupby("GLACIER").agg(
        POINT_LAT=("POINT_LAT", "mean"),
        POINT_LON=("POINT_LON", "mean"),
        RGI_REGION=("RGI_REGION", "first"),
    ))
    glacier_info[nmeas_col] = df_stakes.groupby("GLACIER").size()

    print(f"[WGMS Global] {len(glacier_info)} glaciers | "
          f"{int(glacier_info[nmeas_col].sum()):,} total stake measurements")
    return glacier_info

In [ ]:
glacier_info_world = build_wgms_global_glacier_info(cfg)

fig, ax, df = plot_wgms_world_map(
    glacier_info=glacier_info_world,
    sizes=(20, 1000),
    size_legend_values=(10, 100, 1000),
    figsize=(16, 8),
    point_color="#b2182b",
)
fig.savefig("figures/paperTF/fig_wgms_world_map.pdf",
            dpi=300,
            bbox_inches="tight")

In [ ]:
df_stakes = pd.read_csv(
    os.path.join(cfg.dataPath, 'WGMS/raw/mass_balance_point.csv'))

df_RGIId = pd.read_csv(cfg.dataPath + 'WGMS/raw/glacier.csv')

# Create a mapping dictionary from gtng_region to all countries
country_to_gtng_region_map = df_RGIId.groupby(
    'gtng_region')['country'].unique().to_dict()
country_to_gtng_region_map

# Create a mapping dictionary from id to rgi60_ids
id_to_rgi_map = dict(zip(df_RGIId['id'], df_RGIId['rgi60_ids']))

# Create a mapping dictionary from id to gtng_region
id_to_gtng_region_map = dict(zip(df_RGIId['id'], df_RGIId['gtng_region']))

# Add the RGIId column to the filtered DataFrame using glacier_id instead of id
df_stakes['RGIId'] = df_stakes['glacier_id'].map(id_to_rgi_map)
df_stakes['gtng_region'] = df_stakes['glacier_id'].map(id_to_gtng_region_map)

print('Unique countries in df_stakes:', df_stakes['country'].unique())
print('Unique gtng_regions in df_stakes:', df_stakes['gtng_region'].unique())
df_stakes.columns

In [ ]:
df_stakes.year.min()

In [ ]:
df_stakes.groupby('country').count()